# Qualitative case review

Prompts whose score rose (or fell) most between two model versions, with both responses. The paper's examples come from
`gemini-2.5-flash-lite -> gemini-2.5-pro` (stereotype proxy, HolisticBias) and
`claude-sonnet-4-20250514 -> claude-sonnet-4-5-20250929` (sentiment, CrowS-Pairs).

Needs `outputs/results/scored_responses.csv` (run `python scripts/score.py`). It contains model
responses, so clear outputs before committing this notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
from src.config import SCORES_PATH  # noqa: E402

pd.set_option("display.max_colwidth", 300)
scored = pd.read_csv(SCORES_PATH, low_memory=False)


def largest_paired_changes(old_model, new_model, metric, dataset=None, decreases=False, top=5):
    """Prompts with the largest score increase (or decrease) between two models."""
    cols = ["dataset", "prompt_id", "prompt", "response", metric]
    old = scored[scored["model"] == old_model][cols]
    new = scored[scored["model"] == new_model][cols].drop(columns="prompt")
    pairs = old.merge(new, on=["dataset", "prompt_id"], suffixes=("_old", "_new"))
    if dataset:
        pairs = pairs[pairs["dataset"] == dataset]
    pairs["change"] = pairs[f"{metric}_new"] - pairs[f"{metric}_old"]
    return pairs.sort_values("change", ascending=decreases).head(top)


largest_paired_changes("gemini-2.5-flash-lite", "gemini-2.5-pro", "stereotype_score", "holistic_bias")

In [ ]:
largest_paired_changes("claude-sonnet-4-20250514", "claude-sonnet-4-5-20250929", "sentiment_score",
                       "crows_pairs", decreases=True)